# Eventos globales GDELT 2.0 — métricas diarias

**Dataset:** [GDELT 2.0](https://blog.gdeltproject.org/gdelt-2-0-our-global-world-in-realtime/) (Global Database of Events, Language, and Tone) — captura en tiempo casi real eventos reportados por medios de todo el mundo, codificados con el esquema [CAMEO](https://gdeltproject.org/data/documentation/CAMEO.Manual.1.1b3.pdf) (actores, tipo de evento, tono, geolocalización, cantidad de menciones/fuentes/artículos).

GDELT publica un archivo de eventos nuevo cada 15 minutos desde 2015, listado en una [lista maestra](https://data.gdeltproject.org/gdeltv2/masterfilelist.txt) que ya tiene varios millones de líneas. El objetivo de este TP es calcular, para cada día desde el **2 de septiembre de 2026**, la cantidad de eventos y el promedio de artículos/fuentes por evento.

## Flujo del análisis

La lógica reutilizable vive en el paquete [`src/`](./src); este notebook sólo la orquesta:

1. `src.master_list` — recorre la lista maestra en streaming y filtra las URLs de eventos del rango de fechas pedido.
2. `src.fetch` — descarga y parsea un archivo de 15 minutos, leyendo sólo las columnas necesarias.
3. `src.aggregate` — acumula totales diarios (eventos, artículos, fuentes) sin retener los eventos individuales.
4. `src.pipeline` — orquesta la descarga en paralelo y la agregación incremental, y devuelve el DataFrame final.

In [1]:
%pip install -q pandas requests tqdm

import io
import sys
from pathlib import Path

# Permite importar el paquete `src` del TP (la carpeta que contiene este notebook).
TP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
if str(TP_ROOT) not in sys.path:
    sys.path.insert(0, str(TP_ROOT))

import pandas as pd
import requests

from src.config import DATA_PROCESSED, DATA_RAW, START_DATE
from src.fetch import fetch_event_file
from src.master_list import date_from_url, filter_event_urls, iter_master_file_list
from src.pipeline import build_daily_stats


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Acceso eficiente a los datos

La lista maestra tiene una entrada por archivo publicado desde 2015 (eventos, menciones y GKG cada 15 min) — varios millones de líneas y un archivo de texto de gran tamaño. Como sólo hace falta la ventana desde el 2 Sep 2026, cargarla entera en memoria (o peor, descargar todo el histórico) sería un desperdicio enorme de red y memoria. Se optó por un enfoque de **acceso incremental por batches**, con varias decisiones concretas:

- **Streaming de la lista maestra** (`iter_master_file_list`): se lee con `requests.get(..., stream=True)` línea por línea, sin materializar el archivo completo en memoria, y se descartan al vuelo las líneas que no interesan.
- **Filtrado por tipo y fecha** (`filter_event_urls`): de los 3 archivos que se publican cada 15 min (eventos, menciones, GKG) sólo se necesita el de eventos (`.export.CSV.zip`); y sólo los publicados a partir del 2 Sep 2026. Esto reduce la lista maestra completa a unos pocos cientos de URLs.
- **Poda de columnas**: el archivo de eventos tiene 61 columnas sin encabezado; para las métricas pedidas alcanza con 2 (`NumSources`, `NumArticles`), que se leen con `pd.read_csv(..., usecols=...)` en vez de parsear las 61.
- **Descarga en paralelo** (`ThreadPoolExecutor`): descargar cada archivo es I/O-bound (se espera a la red, no a la CPU), así que varios workers concurrentes aceleran mucho el total sin necesitar más memoria.
- **Agregación incremental por batches** (`DailyAccumulator`): cada archivo de 15 min (unos pocos miles de eventos) se suma a un acumulador de totales por día y se descarta inmediatamente. En ningún momento se concatenan los eventos crudos de varios días en un único DataFrame — el uso de memoria queda acotado por la cantidad de *días* analizados, no por la cantidad de eventos.
- **Caché local** (`cache_dir=data/raw/`): los ZIP ya descargados se guardan en disco (carpeta ignorada por git) para no volver a bajarlos si se re-ejecuta el notebook.

Se descartó usar el dataset público de GDELT en BigQuery: es la opción más eficiente para consultas históricas de gran volumen, pero acá alcanza con ~9 días (unos cientos de archivos) y el acceso directo por HTTP no requiere credenciales ni configurar un proyecto de GCP.

In [2]:
urls = list(filter_event_urls(iter_master_file_list(), start_date=START_DATE))
print(f"Archivos de eventos desde {START_DATE}: {len(urls)}")
print(urls[0])
print(urls[-1])

Archivos de eventos desde 20260902: 858
http://data.gdeltproject.org/gdeltv2/20260902000000.export.CSV.zip
http://data.gdeltproject.org/gdeltv2/20260910221500.export.CSV.zip


### Por qué agrupar por la fecha del archivo (`DATEADDED`) y no por `SQLDATE`

Cada evento trae un `SQLDATE`, la fecha en que CAMEO estima que *ocurrió* el evento — no necesariamente la fecha en que se lo registró. Un mismo archivo de 15 min puede incluir eventos con `SQLDATE` de días o meses anteriores, cuando un artículo publicado ese día hace referencia a un hecho pasado. Se puede comprobar leyendo esa columna en un archivo real:

In [3]:
url_muestra = urls[0]
zip_bytes = requests.get(url_muestra, timeout=30).content
sqldate_muestra = pd.read_csv(
    io.BytesIO(zip_bytes), compression="zip", sep="\t",
    header=None, usecols=[1], names=["SQLDATE"],
)
print(f"Archivo: {Path(url_muestra).name} -> fecha de ingesta: {date_from_url(url_muestra)}")
sqldate_muestra["SQLDATE"].value_counts().head()

Archivo: 20260902000000.export.CSV.zip -> fecha de ingesta: 20260902


SQLDATE
20260902    2366
20260826      23
20260803      15
20250902       4
Name: count, dtype: int64

La gran mayoría de los eventos del archivo tienen `SQLDATE` igual a la fecha del archivo, pero no todos. Como el enunciado pide los eventos *registrados* desde el 2 Sep 2026 (y no los que *ocurrieron* desde esa fecha), agrupar por `DATEADDED` -equivalente a la fecha del archivo- es la interpretación correcta, además de la más eficiente: ya se está filtrando por esa fecha al elegir qué archivos descargar (celda anterior), así que no hace falta leer ninguna columna de fecha por evento. Por eso `src.fetch.fetch_event_file` sólo lee `NumSources` y `NumArticles`, y `src.pipeline.build_daily_stats` asocia cada archivo a su día con `date_from_url`.

## 2. Cálculo de las métricas diarias

`build_daily_stats` combina los pasos anteriores: filtra las URLs desde `START_DATE = "20260902"`, las descarga en paralelo y agrega los totales por día.

In [4]:
df_diario = build_daily_stats(start_date=START_DATE, max_workers=16)
df_diario

Archivos GDELT:   0%|          | 0/858 [00:00<?, ?it/s]

Archivos GDELT:  12%|█▏        | 101/858 [00:00<00:00, 986.42it/s]

Archivos GDELT:  23%|██▎       | 200/858 [00:00<00:02, 245.51it/s]

Archivos GDELT:  29%|██▉       | 251/858 [00:01<00:02, 208.20it/s]

Archivos GDELT:  33%|███▎      | 286/858 [00:01<00:02, 193.83it/s]

...

Archivos GDELT:  95%|█████████▌| 816/858 [00:04<00:00, 140.38it/s]

Archivos GDELT:  97%|█████████▋| 834/858 [00:05<00:00, 149.03it/s]

Archivos GDELT:  99%|█████████▉| 852/858 [00:05<00:00, 155.57it/s]

Archivos GDELT: 100%|██████████| 858/858 [00:05<00:00, 166.59it/s]

,cantidad_eventos,promedio_articulos_por_evento,promedio_fuentes_por_evento
fecha,,,
2026-09-02,120105,4.642246,1.048508
2026-09-03,117020,4.630003,1.045454
2026-09-04,107037,4.686781,1.047012
2026-09-05,66878,4.640794,1.039923
2026-09-06,60454,4.574304,1.043835
2026-09-07,86043,4.677684,1.040259
2026-09-08,114917,4.601887,1.046956
2026-09-09,119065,4.625264,1.041893
2026-09-10,111639,4.607556,1.044259


## 3. Resultados

El resultado es un DataFrame de Pandas indexado por fecha, con una fila por día desde el 2 Sep 2026 y las 3 métricas pedidas. *Nota:* el último día puede estar incompleto si el notebook se corre antes de que termine de publicarse (GDELT actualiza cada 15 min).

In [5]:
resumen = df_diario.rename(columns={
    "cantidad_eventos": "Cantidad de eventos",
    "promedio_articulos_por_evento": "Promedio de artículos por evento",
    "promedio_fuentes_por_evento": "Promedio de fuentes por evento",
})
resumen

,Cantidad de eventos,Promedio de artículos por evento,Promedio de fuentes por evento
fecha,,,
2026-09-02,120105,4.642246,1.048508
2026-09-03,117020,4.630003,1.045454
2026-09-04,107037,4.686781,1.047012
2026-09-05,66878,4.640794,1.039923
2026-09-06,60454,4.574304,1.043835
2026-09-07,86043,4.677684,1.040259
2026-09-08,114917,4.601887,1.046956
2026-09-09,119065,4.625264,1.041893
2026-09-10,111639,4.607556,1.044259


In [6]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
salida_csv = DATA_PROCESSED / "gdelt_metricas_diarias.csv"
df_diario.to_csv(salida_csv)
print(f"Archivo guardado: {salida_csv}")

Archivo guardado: C:\workspace\IID_MP\TP\tp1\data\processed\gdelt_metricas_diarias.csv


## 4. Por qué los promedios son tan parecidos entre días

Los promedios de `promedio_articulos_por_evento` (~4.6) y `promedio_fuentes_por_evento` (~1.04) resultaron casi idénticos todos los días. Esto no es un error del pipeline: es una propiedad estructural de cómo GDELT arma los eventos, que se puede confirmar mirando la distribución completa de estos dos campos para todo el período. La reconstruimos reutilizando el caché local de `data/raw/` (los ZIP de la sección 2), así que esto no descarga nada nuevo de la red.

In [7]:
# Sólo 2 columnas livianas (NumSources, NumArticles) para ~900 mil eventos: unos pocos MB.
# A diferencia de build_daily_stats, acá sí conviene tener todo junto porque el objetivo
# es mirar la distribución completa, no sólo sumas por día.
eventos = pd.concat(
    [fetch_event_file(url, cache_dir=DATA_RAW) for url in urls],
    ignore_index=True,
)
print(f"Eventos totales ({START_DATE} en adelante): {len(eventos):,}")

Eventos totales (20260902 en adelante): 903,158


In [8]:
print("--- NumSources ---")
print(eventos["NumSources"].describe())
print(f"Eventos con NumSources == 1: {(eventos['NumSources'] == 1).mean():.1%}")

--- NumSources ---
count    903158.000000
mean          1.044606
std           0.394874
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          20.000000
Name: NumSources, dtype: float64
Eventos con NumSources == 1: 97.5%


In [9]:
print("--- NumArticles ---")
print(eventos["NumArticles"].describe())

q99 = eventos["NumArticles"].quantile(0.99)
top_1pct = eventos[eventos["NumArticles"] >= q99]
print(f"\nMediana: {eventos['NumArticles'].median():.0f} artículos por evento")
print(f"Umbral p99: {q99:.0f} artículos")
print(
    f"Ese ~1% de eventos más cubiertos explica el "
    f"{top_1pct['NumArticles'].sum() / eventos['NumArticles'].sum():.1%} de los artículos "
    "totales (no son unos pocos outliers los que arrastran el promedio)."
)

--- NumArticles ---
count    903158.000000
mean          4.632997
std           4.021480
min           1.000000
25%           2.000000
50%           4.000000
75%           6.000000
max         260.000000
Name: NumArticles, dtype: float64

Mediana: 4 artículos por evento
Umbral p99: 15 artículos
Ese ~1% de eventos más cubiertos explica el 5.5% de los artículos totales (no son unos pocos outliers los que arrastran el promedio).


**Lectura de los resultados:**

- **`NumSources` está pegado a 1.** En la gran mayoría de los eventos (>95%) hay una única fuente distinta detrás. GDELT define un evento a un nivel muy granular -actor1 × actor2 × código CAMEO × fecha × ubicación-, y a ese nivel de detalle casi todas las combinaciones sólo las reporta un dominio de noticias dentro de la ventana de 15 min en que GDELT las captura. Es la "cola larga" típica de GDELT: cientos de miles de combinaciones posibles de actor/acción/lugar, cubiertas por un conjunto de medios más o menos constante.
- **`NumArticles` es mayor y algo más variable, pero no está dominado por outliers.** La mediana es 4, y el 1% de eventos más cubiertos explica sólo una fracción chica del total de artículos -no son unos pocos eventos virales los que arrastran el promedio-. `NumArticles` cuenta *artículos* (URLs), no fuentes: un mismo medio suele republicar la misma nota de agencia (AP, Reuters, etc.) en varias URLs, y varios medios chicos que levantan el mismo cable matchean el mismo evento. Eso separa el promedio de artículos del promedio de fuentes.
- **Por qué es tan estable entre días.** Estos son promedios sobre decenas o cientos de miles de eventos por día. A esa escala, la relación "artículos/fuentes por evento" es una propiedad del *pipeline de extracción* de GDELT -qué tan grande es el conjunto de medios que monitorea, y cómo deduplica coberturas- y no depende de qué tan importante fue la noticia de ese día en particular. Por ley de los grandes números, el promedio converge a un valor muy similar todos los días aunque el contenido de las noticias cambie por completo. Esto no invalida la métrica pedida por el enunciado, sólo explica por qué no se espera que varíe mucho día a día -a diferencia de `cantidad_eventos`, que sí refleja volumen real de cobertura y varía notoriamente (ver fines de semana vs. días de semana en la tabla de la sección 3).